# Master Kashmiri NLLB Translation Fine-Tuning (320,000+ Sentence Pairs)

This notebook fine-tunes **Meta NLLB-200** on a massive combined Kashmiri corpus including:
- **SMUQamar 270K** Kashmiri-English parallel dataset
- **Omarrran 49K** Kashmiri-English translation corpus
- **Kaggle Clean & Expanded Datasets**
- **JKBOSE Kashmiri School Textbooks & In-House Grammar Lessons**

### Instructions:
1. **Accelerator**: Select **GPU P100** or **GPU T4 x2**.
2. **Internet**: Toggle **Internet ON**.
3. **Datasets**: Attach `sameerahmadnajar/kashmiri-clean-parallel-dataset` and `sameerahmadnajar/kashmiri-expanded-parallel-dataset`.
4. Click **Run All**.


In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
!pip -q install 'transformers==4.48.3' 'peft==0.14.0' 'accelerate==1.3.0' 'datasets>=3.2' 'evaluate>=0.4' sacrebleu sentencepiece huggingface_hub


In [ ]:
# 2. Clone/Update Cipher Repository with Master NLLB Training Pipeline
import subprocess
from pathlib import Path

repo = Path("/kaggle/working/cipher")
if not repo.exists():
    print("[*] Cloning cipher repository...")
    subprocess.run(["git", "clone", "https://github.com/sameerahmad86232/cipher.git", str(repo)], check=True)
else:
    print("[*] Pulling latest cipher repository...")
    subprocess.run(["git", "pull"], cwd=str(repo), check=True)

script = repo / "scripts/finetune-nllb-master.py"
assert script.exists(), f"Missing {script}"
print("[+] Training script ready at:", script)


In [ ]:
# 3. Launch Master NLLB Kashmiri Translation Training
import os, subprocess

env = os.environ.copy()
env["BASE_MODEL"] = "facebook/nllb-200-distilled-600M"
env["NUM_EPOCHS"] = "3"
env["BATCH_SIZE"] = "2"
env["GRAD_ACCUM"] = "8"
env["LEARNING_RATE"] = "1e-4"
env["OUTPUT_DIR"] = "/kaggle/working/kashmiri-nllb-master"
env["HF_REPO_ID"] = "Sameer0313/kashmiri-nllb-lora"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# env["HF_TOKEN"] = "your_hf_token_here"

subprocess.run(["python", "scripts/finetune-nllb-master.py"], cwd=str(repo), env=env, check=True)


In [ ]:
# 4. Check Output & Generated Translation Report
from pathlib import Path
out = Path('/kaggle/working/kashmiri-nllb-master')
report_file = out / 'training-report.json'
if report_file.exists():
    print('\n================ MASTER TRAINING REPORT ================\n')
    print(report_file.read_text())
    print('========================================================')
zip_file = Path('/kaggle/working/kashmiri-nllb-master-adapter.zip')
if zip_file.exists():
    print(f'Download master adapter: {zip_file.name} ({zip_file.stat().st_size / 1024 / 1024:.2f} MB)')
